In [12]:
import sys
sys.path.append('..')  # supaya bisa import dari folder core

import cv2
import matplotlib.pyplot as plt
from core.preprocessor import Preprocessor

# Test dengan foto apapun dulu
prep = Preprocessor()
stages = prep.run("path/foto_test.jpg", debug=True)

# Visualisasi semua tahap
titles = ["Original", "Grayscale", "Histogram EQ", "Denoised (final)"]
images = list(stages.values())

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img, title in zip(axes, images, titles):
    if len(img.shape) == 3:
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    else:
        ax.imshow(img, cmap='gray')
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()
plt.savefig("preprocessing_result.png", dpi=150)
plt.show()

ValueError: Gagal membaca gambar. Pastikan format JPG/PNG.

In [11]:
%pip install opencv-python matplotlib

  Using cached matplotlib-3.10.9-cp312-cp312-win_amd64.whl.metadata (52 kB)
  Using cached contourpy-1.3.3-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.5.0-cp312-cp312-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.2.0-cp312-cp312-win_amd64.whl.metadata (9.0 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached matplotlib-3.10.9-cp312-cp312-win_amd64.whl (8.2 MB)
Using cached contourpy-1.3.3-cp312-cp312-win_amd64.whl (226 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ------------------------------- -------- 1.8/2.3 MB 9.1 MB/s eta 0:00:01
   ---------------------------------------- 2.3/2.3 MB 8.3 MB/s eta 0:00:00
Using cached kiwisolver-1.5.0-cp312-cp312-win_amd64.whl (73 kB)
Using cached pillow-12.2.0-cp312-cp312-win_amd64.whl (7.1 MB)
Using cached pyparsing-3.3.2-py3-none-an


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from core.preprocessor import Preprocessor
from core.face_detector import FaceDetector

prep = Preprocessor()
detector = FaceDetector()

# Load & preprocess
img_bgr = prep.load_image("path/foto_test.jpg")
stages  = prep.run(img_bgr, debug=True)
gray    = stages["denoised"]

# Validasi foto dulu
is_valid, msg = detector.validate_photo(gray)
print(f"Validasi: {msg}")

if is_valid:
    result, err = detector.detect(gray, debug=True)
    if err:
        print("Error:", err["error"])
    else:
        print(f"Wajah ditemukan: bbox = {result['bbox']}")
        
        # Visualisasi bounding box di atas foto asli
        img_annotated = detector.draw_bbox(
            stages["original"], result["bbox"]
        )
        
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(cv2.cvtColor(stages["original"], cv2.COLOR_BGR2RGB))
        axes[0].set_title("Original")
        axes[0].axis("off")
        axes[1].imshow(cv2.cvtColor(img_annotated, cv2.COLOR_BGR2RGB))
        axes[1].set_title("Face Detected")
        axes[1].axis("off")
        plt.tight_layout()
        plt.show()


In [ ]:
from core.landmark import LandmarkDetector
import matplotlib.pyplot as plt
import cv2

lm = LandmarkDetector()

# Pakai hasil dari cell sebelumnya
bbox   = result["bbox"]      # dari FaceDetector
result_lm = lm.detect(gray, bbox, debug=True)

# Tampilkan foto dengan semua 68 titik
plt.figure(figsize=(7, 7))
plt.imshow(cv2.cvtColor(result_lm["annotated"], cv2.COLOR_BGR2RGB))
plt.title("68 Facial Landmarks")
plt.axis("off")
plt.show()

# Cetak fitur geometri
features = lm.extract_features(result_lm["points"])
print("\nFitur geometri wajah:")
for k, v in features.items():
    print(f"  {k:20s} : {v}")
